# pyLSTemp - Exemplo de `emissivity(...)`

Este notebook demonstra como calcular emissividade de superficie para os fluxos termais da pyLSTemp.

A emissividade e calculada a partir de um NDVI previamente estimado. Alguns metodos usam apenas NDVI; outros tambem usam a banda Red.

## 1. Instalacao

In [ ]:
# Em Colab/Jupyter, descomente se precisar instalar.
# !pip install --quiet --upgrade git+https://github.com/daciocambraia/pyLSTemp.git rasterio

## 2. Importacoes

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from pylstemp import spectral_index, emissivity, list_algorithms

import pylstemp
print("pyLSTemp version:", pylstemp.__version__)

## 3. Conferir metodos de emissividade

In [ ]:
catalog = list_algorithms()
print(list(catalog["emissivity"].keys()))

## 4. Ler bandas Red e NIR

In [ ]:
data_dir = Path("data")

red_path = data_dir / "LC82210712016107LGN01_B4.tif"
nir_path = data_dir / "LC82210712016107LGN01_B5.tif"

with rasterio.open(red_path) as src:
    red = src.read(1).astype(float)
    profile = src.profile.copy()

with rasterio.open(nir_path) as src:
    nir = src.read(1).astype(float)

mask = (red == 0) | (nir == 0)

print("Red shape:", red.shape)
print("NIR shape:", nir.shape)
print("Pixels invalidos:", int(mask.sum()))

## 5. Calcular NDVI

A emissividade recebe `ndvi_image`. Por isso, calculamos o NDVI antes.

In [ ]:
ndvi_image = spectral_index(index="ndvi", nir=nir, red=red, mask=mask)
print("NDVI min/max/mean:", np.nanmin(ndvi_image), np.nanmax(ndvi_image), np.nanmean(ndvi_image))

## 6. Calcular emissividade para single-channel

`avdan-2016` e indicado para o fluxo single-channel. Ele retorna a emissividade usada no workflow da banda 10.

In [ ]:
emissivity_10_avdan = emissivity(
    ndvi_image=ndvi_image,
    band="band_10",
    emissivity_method="avdan-2016",
)

print("Avdan Band 10 min/max/mean:", np.nanmin(emissivity_10_avdan), np.nanmax(emissivity_10_avdan), np.nanmean(emissivity_10_avdan))

## 7. Calcular emissividade para split-window

Para split-window, prefira metodos que geram emissividade separada para as bandas 10 e 11, como `gopinadh-2018` ou `xiaolei-2014`.

In [ ]:
emissivity_10_gopinadh = emissivity(ndvi_image, band="band_10", emissivity_method="gopinadh-2018")
emissivity_11_gopinadh = emissivity(ndvi_image, band="band_11", emissivity_method="gopinadh-2018")

emissivity_10_xiaolei = emissivity(ndvi_image, band="band_10", band_4_red=red, emissivity_method="xiaolei-2014")
emissivity_11_xiaolei = emissivity(ndvi_image, band="band_11", band_4_red=red, emissivity_method="xiaolei-2014")

print("Gopinadh Band 10 mean:", np.nanmean(emissivity_10_gopinadh))
print("Gopinadh Band 11 mean:", np.nanmean(emissivity_11_gopinadh))
print("Xiaolei Band 10 mean:", np.nanmean(emissivity_10_xiaolei))
print("Xiaolei Band 11 mean:", np.nanmean(emissivity_11_xiaolei))

## 8. Visualizar emissividade

In [ ]:
def show_raster(image, title, cmap="viridis"):
    valid = image[np.isfinite(image)]
    vmin, vmax = np.nanpercentile(valid, [2, 98])
    plt.figure(figsize=(8, 6))
    plt.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(label="Emissivity")
    plt.title(title)
    plt.axis("off")
    plt.show()

show_raster(emissivity_10_avdan, "Emissivity Band 10 - Avdan 2016")
show_raster(emissivity_10_gopinadh, "Emissivity Band 10 - Gopinadh 2018")
show_raster(emissivity_11_gopinadh, "Emissivity Band 11 - Gopinadh 2018")

## 9. Salvar resultado opcionalmente

In [ ]:
# output_path = Path("emissivity_band_10_avdan.tif")
# output_profile = profile.copy()
# output_profile.update(dtype="float32", nodata=np.nan, count=1)
#
# with rasterio.open(output_path, "w", **output_profile) as dst:
#     dst.write(emissivity_10_avdan.astype("float32"), 1)
#
# print("Arquivo salvo em:", output_path)